In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import json

In [2]:
image = cv2.imread('sample_input_output/images/image1.jpg')
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


In [4]:
edged = cv2.Canny(image=gray, threshold1=30, threshold2=200)
cv2.imshow('Canny Edges', edged)
cv2.waitKey(0)
cv2.destroyAllWindows() 

In [5]:
contours, hierarchy = cv2.findContours(edged, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
contour_image = image.copy()
largest_contour = max(contours, key=cv2.contourArea)
x, y, w, h = cv2.boundingRect(largest_contour)
h_img, w_img = gray.shape[:2]

ROI = gray[max(0, y-5): min(h_img, y+h+5), max(0, x-5): min(w_img, x+w+5)]
edged_image = edged[max(0, y-5): min(h_img, y+h+5), max(0, x-5): min(w_img, x+w+5)]

cv2.imwrite('sample_input_output/images/grid.png', ROI)
cv2.imwrite('sample_input_output/images/edged.png', edged_image)

cv2.rectangle(image, (x, y), (x + w, y + h), (36,255,12), 2)
cv2.drawContours(contour_image, [largest_contour], -1, (0,255,0), 2)

cv2.imshow("Contours Debug", contour_image)
cv2.waitKey(0)
cv2.destroyAllWindows()

cv2.imshow("bounding", image)
cv2.waitKey(0)
cv2.destroyAllWindows()


In [6]:
def sort_cells_into_grid(image_plus_center, tolerance=15):
    cells = sorted(image_plus_center, key=lambda c: c[0][1])
    rows = []
    current_row = [cells[0]]

    for cell in cells[1:]:
        _, cy = cell[0]
        _, prev_cy = current_row[-1][0]

        if abs(cy - prev_cy) <= tolerance:
            current_row.append(cell)
        else:
            current_row.sort(key=lambda c: c[0][0])
            rows.append(current_row)
            current_row = [cell]

    current_row.sort(key=lambda c: c[0][0])
    rows.append(current_row)

    return [cell for row in rows for cell in row]

In [12]:
def preprocess(image, iterations, size):
    data = {}
    image_plus_center = []
    num = 0
    grid_img = cv2.resize(image, (450, 450))  # resize for better detection

    gray = cv2.cvtColor(grid_img, cv2.COLOR_BGR2GRAY)
    thresh = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY_INV, 11, 2)
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, size)
    fixed = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel, iterations=iterations)

    cv2.imshow("fixed", fixed)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    contours, _ = cv2.findContours(fixed, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    shape = grid_img.shape
    expected_area = (shape[0] * shape[1]) / 81
    min_area = expected_area * 0.5  
    max_area = expected_area * 1.2
    contour_image_grid = grid_img.copy()
    for c in contours:
        area = cv2.contourArea(c)
        cv2.drawContours(contour_image_grid, [c], -1, (0, 255, 0), 2)
        if min_area < area < max_area:
            print(area)
            x, y, w, h = cv2.boundingRect(c)
            cv2.rectangle(grid_img, (x, y), (x + w, y + h), (36, 255, 12), 2)
            ROI = grid_img[y:y+h, x:x+w]
            cell_gray = cv2.cvtColor(ROI, cv2.COLOR_BGR2GRAY)
            cell_blur = cv2.bilateralFilter(cell_gray, 5, 75, 75)
            thrs_cell = cv2.threshold(cell_blur, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
            contours, _ = cv2.findContours(thrs_cell, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            if len(contours) == 0:
                resized_cell = np.zeros((28, 28), dtype=np.uint8)
            else:
                resized_cell = cv2.resize(thrs_cell, (28, 28))

            image_plus_center.append(((x + w//2, y + h//2), resized_cell))
            num = num + 1

    cv2.imshow("Contours Debug", contour_image_grid)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    cv2.imshow("Bounding Boxes", grid_img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
    return num, contour_image_grid, grid_img, image_plus_center, data



In [13]:
grid_image = cv2.imread('data/kaggle_dataset/v1_training/v1_training/image9.jpg')
for i in range(1, 9):
    if i == 1:
        size = (2, 2)
        iterations = 1
        num, grid_img, contour_image_grid, image_plus_center, data = preprocess(grid_image, iterations, size)
        
        if num == 81:
            sorted_cells = sort_cells_into_grid(image_plus_center, tolerance=15)
           
            for idx, (center, cell_img) in enumerate(sorted_cells):
                cv2.imwrite(f'app/digit_images/cell_{idx}.png', cell_img)
                data[f"cell_{idx}"] = {"center": [center[0], center[1]]}
            with open("app/solve/cells_data.json", "w") as f:
                json.dump(data, f, indent=4)

            cv2.imshow('contour_image_grid', contour_image_grid)
            cv2.imshow('grid_img', grid_img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
            break
    elif i == 2:
        size = (2, 2)
        iterations = 2
        num, grid_img, contour_image_grid, image_plus_center, data = preprocess(grid_image, iterations, size)
        if num == 81:
            sorted_cells = sort_cells_into_grid(image_plus_center, tolerance=15)
           
            for idx, (center, cell_img) in enumerate(sorted_cells):
                cv2.imwrite(f'app/digit_images/cell_{idx}.png', cell_img)
                data[f"cell_{idx}"] = {"center": [center[0], center[1]]}
            with open("app/solve/cells_data.json", "w") as f:
                json.dump(data, f, indent=4)

            cv2.imshow('contour_image_grid', contour_image_grid)
            cv2.imshow('grid_img', grid_img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
            break

    elif i == 3:
        size = (2, 2)
        iterations = 3
        num, grid_img, contour_image_grid, image_plus_center, data = preprocess(grid_image, iterations, size)
        if num == 81:
            sorted_cells = sort_cells_into_grid(image_plus_center, tolerance=15)
           
            for idx, (center, cell_img) in enumerate(sorted_cells):
                cv2.imwrite(f'app/digit_images/cell_{idx}.png', cell_img)
                data[f"cell_{idx}"] = {"center": [center[0], center[1]]}
            with open("app/solve/cells_data.json", "w") as f:
                json.dump(data, f, indent=4)

            cv2.imshow('contour_image_grid', contour_image_grid)
            cv2.imshow('grid_img', grid_img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
            break

    elif i == 4:
        size = (3, 3)
        iterations = 1
        num, grid_img, contour_image_grid, image_plus_center, data = preprocess(grid_image, iterations, size)
        if num == 81:
            sorted_cells = sort_cells_into_grid(image_plus_center, tolerance=15)
           
            for idx, (center, cell_img) in enumerate(sorted_cells):
                cv2.imwrite(f'app/digit_images/cell_{idx}.png', cell_img)
                data[f"cell_{idx}"] = {"center": [center[0], center[1]]}
            with open("app/solve/cells_data.json", "w") as f:
                json.dump(data, f, indent=4)

            cv2.imshow('contour_image_grid', contour_image_grid)
            cv2.imshow('grid_img', grid_img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
            break

    elif i == 5:
        size = (3, 3)
        iterations = 2
        num, grid_img, contour_image_grid, image_plus_center, data = preprocess(grid_image, iterations, size)
        if num == 81:
            sorted_cells = sort_cells_into_grid(image_plus_center, tolerance=15)
           
            for idx, (center, cell_img) in enumerate(sorted_cells):
                cv2.imwrite(f'app/digit_images/cell_{idx}.png', cell_img)
                data[f"cell_{idx}"] = {"center": [center[0], center[1]]}
            with open("app/solve/cells_data.json", "w") as f:
                json.dump(data, f, indent=4)

            cv2.imshow('contour_image_grid', contour_image_grid)
            cv2.imshow('grid_img', grid_img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
            break

    elif i == 6:
        size = (3, 3)
        iterations = 3
        num, grid_img, contour_image_grid, image_plus_center, data = preprocess(grid_image, iterations, size)
        if num == 81:
            sorted_cells = sort_cells_into_grid(image_plus_center, tolerance=15)
           
            for idx, (center, cell_img) in enumerate(sorted_cells):
                cv2.imwrite(f'app/digit_images/cell_{idx}.png', cell_img)
                data[f"cell_{idx}"] = {"center": [center[0], center[1]]}
            with open("app/solve/cells_data.json", "w") as f:
                json.dump(data, f, indent=4)

            cv2.imshow('contour_image_grid', contour_image_grid)
            cv2.imshow('grid_img', grid_img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
            break

    elif i == 7:
        size = (5, 5)
        iterations = 1
        num, grid_img, contour_image_grid, image_plus_center, data = preprocess(grid_image, iterations, size)
        if num == 81:
            sorted_cells = sort_cells_into_grid(image_plus_center, tolerance=15)
           
            for idx, (center, cell_img) in enumerate(sorted_cells):
                cv2.imwrite(f'app/digit_images/cell_{idx}.png', cell_img)
                data[f"cell_{idx}"] = {"center": [center[0], center[1]]}
            with open("app/solve/cells_data.json", "w") as f:
                json.dump(data, f, indent=4)

            cv2.imshow('contour_image_grid', contour_image_grid)
            cv2.imshow('grid_img', grid_img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
            break

    elif i == 8:
        size = (5, 5)
        iterations = 2
        num, grid_img, contour_image_grid, image_plus_center, data = preprocess(grid_image, iterations, size)
        if num == 81:
            sorted_cells = sort_cells_into_grid(image_plus_center, tolerance=15)
           
            for idx, (center, cell_img) in enumerate(sorted_cells):
                cv2.imwrite(f'app/digit_images/cell_{idx}.png', cell_img)
                data[f"cell_{idx}"] = {"center": [center[0], center[1]]}
            with open("app/solve/cells_data.json", "w") as f:
                json.dump(data, f, indent=4)

            cv2.imshow('contour_image_grid', contour_image_grid)
            cv2.imshow('grid_img', grid_img)
            cv2.waitKey(0)
            cv2.destroyAllWindows()
            break
        


1668.0
1615.0
1649.0
1645.5
1695.5
1589.0
1576.0
1510.5
1490.5
1564.0
1652.0
1649.0
1575.5
1632.5
1622.0
1543.5
1603.5
1520.5
1595.0
1704.0
1635.5
1581.5
1591.5
1638.0
1719.0
1693.5
1699.0
1481.0
1517.5
1515.0
1558.0
1581.5
1600.5
1573.5
1636.0
1576.5
1547.5
1552.0
1596.0
1535.0
1616.0
1659.0
1600.5
1664.5
1624.0
1590.5
1591.0
1641.0
1618.5
1657.5
1694.0
1639.5
1674.0
1638.0
1517.0
1543.5
1568.0
1507.0
1568.0
1604.5
1558.0
1569.5
1553.0
1547.5
1590.0
1612.0
1558.5
1628.5
1677.5
1604.0
1664.0
1604.0
1555.0
1565.0
1670.5
1567.0
1672.5
1667.5
1615.0
1620.0
1564.5


In [8]:
import os

folder_path = "app/digit_images"
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif")

for filename in os.listdir(folder_path):
    if filename.lower().endswith(image_extensions):
        file_path = os.path.join(folder_path, filename)
        os.remove(file_path)
        # print(f"Deleted: {filename}")

print("Done")

Done
